# Judge a pool with llmjudge

Runtime → Change runtime type → **A100** or **L4**, then run the cells top to bottom.

**Code comes from GitHub. Drive holds only data** — the items file going in, the
results coming out. Nothing else is uploaded, and the runtime can die without losing
anything.

Two Colab Secrets (the key icon in the sidebar, then toggle notebook access):

| secret | what |
|---|---|
| `GH_TOKEN` | fine-grained PAT, **Contents: Read-only**, scoped to this one repo |
| `HF_TOKEN` | for the MedGemma weights |

Neither is ever typed into a cell, so a shared notebook carries no credential.

Before the first run, put one items file on Drive:

```bash
llmjudge make-items --spec configs/pool.diabetes130.toml --pool pilot \
    --root /path/to/your/runs --out items-pilot.jsonl
```

and upload it to `MyDrive/judge/` at drive.google.com. 114 KB for the pilot.

## 1. Install, and mount Drive

In [ ]:
REPO  = 'your-org/llmjudge'                    # <- set once
TAG   = 'v0.1.0'
DRIVE = '/content/drive/MyDrive/judge'

from google.colab import drive, userdata
drive.mount('/content/drive')

!pip -q install "git+https://{userdata.get('GH_TOKEN')}@github.com/{REPO}.git@{TAG}"

import llmjudge; print('llmjudge', llmjudge.__version__ if hasattr(llmjudge, '__version__') else '', llmjudge.__file__)

## 2. The pool

Straight off Drive. The judge reads it from there; only the results are written to
`/content` and mirrored back.

In [ ]:
ITEMS = f'{DRIVE}/items-pilot.jsonl'

import collections, json
rows = [json.loads(l) for l in open(ITEMS)]
print(len(rows), 'rows  ', dict(collections.Counter(r.get('group', 'all') for r in rows)))
print(len(rows[0]['fields']), 'columns')

## 3. Serve MedGemma

~10 minutes the first time: the weights download, then vLLM loads them. The script
ships inside the package. It exports `LLMJUDGE_BASE_URL`, `LLMJUDGE_API_KEY` and
`LLMJUDGE_MODEL`, so the judge cell says nothing about the server.

Judging against a vendor API instead? Skip this cell and set those three environment
variables yourself.

In [ ]:
import os, llmjudge
SERVE = os.path.join(os.path.dirname(llmjudge.__file__), 'serve_vllm.py')
%run {SERVE}

## 4. Judge

Results are written to `/content`, where append and fsync mean what they say, and the
tail is copied up to Drive every 100 seconds and once more at the end. Re-running this
cell after a disconnect resumes: rows already on Drive are not re-sent.

Exit code `0` is every planned row recorded, `3` stopped incomplete, `4` judged but not
all on Drive — `4` is the one to care about, and it names where the rows still are.

In [ ]:
from llmjudge.colab import run

exit_code = run(items=ITEMS,
                out=f'{DRIVE}/results/pilot',
                run_tag='pilot-01',
                prompt='c3-reasoning-2',
                guided=False,
                max_tokens=2560)
print('exit', exit_code)

## 5. What it said

In [ ]:
import json

with open(f'{DRIVE}/results/pilot/summary.json') as f:
    s = json.load(f)

print(s['rows'], 'rows,', s['missing'], 'missing,',
      s['parse_errors'], 'parse errors,', s['errors'] or 'no errors')

for group, g in sorted(s['by_group'].items()):
    print(f"\n{group}  ({g['judged']}/{g['rows']} judged)")
    for label, rate in sorted(g['rates'].items()):
        print(f"   {label:<14} {rate:7.1%}   weighted {g['rates_weighted'][label]:7.1%}")

---

**Sizing a real run.** The last full run did 5,500 rows in 1h50m on one A100 — about
50 rows/minute, p50 latency 20 s. The `full` pool is 55,039 rows, so ~18 hours on one
runtime: that needs either `--mode shard` across two of these notebooks, or several
sessions resuming into the same Drive folder.